# Silver: Join weather and electricity prices
This notebook joins the cleaned hourly electricity prices (`NO2`) with Sola weather observations (`SN44560`) on the UTC hour, tags each row with its weather match status without imputing missing values, and merges the result into `silver.weather_price_hourly`.

### Key design choices in this notebook
* **Left join with electricity prices as the base:** Electricity price series are complete for every hour of the year, whereas physical weather sensors can have occasional outages. Using a `LEFT JOIN` from `electricity_price_hourly` ensures we never drop a price hour due to a missing weather reading.
* **Joining on `time_start_utc`:** Because both sources were normalized to `UTC` in `03_silver_transformations`, joining on `time_start_utc` handles the daylight saving time transition cleanly without duplicate or missing hours.
* **Why we filter prices on `time_start_local`:** When `date_from` is `2025-01-01`, the first local Norwegian hour (`00:00+01:00`) is `2024-12-31 23:00:00 UTC`. Filtering `price_df` on `to_date(time_start_local)` keeps all 24 local hours for each requested date, while leaving `weather_df` unfiltered by local date ensures that if `2024-12-31 23:00:00 UTC` exists in `silver.weather_hourly`, it joins cleanly.
* **No imputation in Silver (`weather_match_status`):** Instead of filling missing weather values with guessed or interpolated numbers, we keep `NULL` values intact and label each hour as `'matched'`, `'partial'`, or `'missing'` so downstream Gold and ML steps can filter explicitly.

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Lock Spark session to UTC and dynamically resolve the active Unity Catalog
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Read runtime parameters from job/notebook widgets (generates a UUID if run_id is empty)
dbutils.widgets.text("date_from", "2025-01-01")
dbutils.widgets.text("date_to", "2025-01-07")
dbutils.widgets.text("run_id", "")

DATE_FROM = dbutils.widgets.get("date_from")
DATE_TO = dbutils.widgets.get("date_to")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

### Load cleaned Silver tables
Loads `silver.electricity_price_hourly` for the requested local date window and `silver.weather_hourly` for station `SN44560` (Sola).


In [0]:
# Filter electricity prices by local Norwegian date so all 24 hours of each date are included
price_df = (
    spark.table(f"{CATALOG}.silver.electricity_price_hourly")
    .filter(F.to_date("time_start_local").between(DATE_FROM, DATE_TO))
)

# Load cleaned Sola weather observations (matched on exact UTC hour in the next cell)
weather_df = (
    spark.table(f"{CATALOG}.silver.weather_hourly")
    .filter(F.col("station_id") == "SN44560")
)

### Left join on UTC hour and tag weather match status
Joins electricity prices with weather observations on `time_start_utc` and classifies each row into `'missing'` (all three weather fields are `NULL`), `'partial'` (temperature or wind speed is `NULL`), or `'matched'` (both temperature and wind speed are present).

In [0]:
# Left join electricity prices with weather on UTC hour so no price hours are lost
joined_df = (
    price_df.alias("price")
    .join(
        weather_df.alias("weather"),
        F.col("price.time_start_utc") == F.col("weather.time_start_utc"),
        "left"
    )
    .select(
        F.col("price.price_area"),
        F.lit("SN44560").alias("station_id"),
        F.col("price.time_start_utc"),
        F.col("price.time_end_utc"),
        F.col("price.time_start_local"),
        F.col("price.nok_per_kwh"),
        F.col("price.eur_per_kwh"),
        F.col("price.exchange_rate"),
        F.col("weather.air_temperature_c"),
        F.col("weather.wind_speed_mps"),
        F.col("weather.precipitation_1h_mm")
    )
    # Tag match completeness explicitly instead of imputing missing sensor values
    .withColumn(
        "weather_match_status",
        F.when(
            F.col("air_temperature_c").isNull()
            & F.col("wind_speed_mps").isNull()
            & F.col("precipitation_1h_mm").isNull(),
            F.lit("missing")
        )
        .when(
            F.col("air_temperature_c").isNull()
            | F.col("wind_speed_mps").isNull(),
            F.lit("partial")
        )
        .otherwise(F.lit("matched"))
    )
    .withColumn("processed_at_utc", F.current_timestamp())
)

### Validate keys and merge into joined Silver table
Verifies that there are no duplicate `(price_area, station_id, time_start_utc)` keys or `NULL` required fields (`time_start_utc`, `nok_per_kwh`), then merges the batch into `silver.weather_price_hourly`.


In [0]:
# Check for duplicate (price_area, station_id, time_start_utc) keys before merging
duplicate_keys = (
    joined_df.groupBy("price_area", "station_id", "time_start_utc")
    .count().filter("count > 1").count()
)

required_errors = joined_df.filter(
    F.col("time_start_utc").isNull()
    | F.col("nok_per_kwh").isNull()
).count()

# Fail fast if the joined DataFrame is empty, has duplicate keys, or is missing price/timestamp
if duplicate_keys or required_errors or joined_df.count() == 0:
    raise ValueError(
        "Joined Silver is empty or contains duplicate/null required keys"
    )

# Upsert into silver.weather_price_hourly on (price_area, station_id, time_start_utc)
(
    DeltaTable.forName(
        spark, f"{CATALOG}.silver.weather_price_hourly"
    ).alias("target")
    .merge(
        joined_df.alias("source"),
        """
        target.price_area = source.price_area
        AND target.station_id = source.station_id
        AND target.time_start_utc = source.time_start_utc
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

### Calculate weather match rate and log data quality checks
Logs two checks to `ops.data_quality_results`:
* **`joined_keys_unique` (`ERROR`):** Confirms zero duplicate business keys in the joined dataset.
* **`weather_match_rate` (`WARNING`):** Checks if at least 95% of hours have `'matched'` weather data. Logged as a `WARNING` so minor sensor gaps (or boundary hours like `2024-12-31 23:00:00 UTC` on a Jan 1 run) are visible in the audit log without stopping the pipeline.

In [0]:
# Calculate the share of hours where both temperature and wind speed matched
total_rows = joined_df.count()
matched_rows = joined_df.filter(
    F.col("weather_match_status") == "matched"
).count()
match_rate = matched_rows / total_rows if total_rows else 0.0

# Log ERROR check for key uniqueness and WARNING check for >= 95% weather match rate
checks = [
    (RUN_ID, "silver_join", "joined_keys_unique", "ERROR",
     duplicate_keys == 0, float(duplicate_keys),
     "Joined business keys must be unique",
     datetime.now(timezone.utc).replace(tzinfo=None)),
    (RUN_ID, "silver_join", "weather_match_rate", "WARNING",
     match_rate >= 0.95, float(match_rate),
     f"Weather match rate is {match_rate:.2%}",
     datetime.now(timezone.utc).replace(tzinfo=None))
]

dq_df = spark.createDataFrame(checks, """
    run_id string, layer string, check_name string, severity string,
    passed boolean, actual_value double, message string,
    checked_at_utc timestamp
""")

# Upsert both check results into ops.data_quality_results
(
    DeltaTable.forName(
        spark, f"{CATALOG}.ops.data_quality_results"
    ).alias("target")
    .merge(
        dq_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.layer = source.layer
        AND target.check_name = source.check_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Weather match rate: {match_rate:.2%}")
display(joined_df.orderBy("time_start_utc").limit(10))